In [ ]:
# !pip install chromadb pypdf langchain-community langchain-huggingface langchain-cohere llama-cpp-python -q

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_classic.retrievers import MergerRetriever, ContextualCompressionRetriever
from langchain_community.document_transformers import LongContextReorder, EmbeddingsRedundantFilter
from langchain_cohere import CohereRerank
from langchain_classic.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
loader_attention = PyPDFLoader(r'papers/Attention_Is_All_You_Need_paper.pdf')
loader_rag = PyPDFLoader(r'papers/RAG-for-NLP_paper.pdf')

In [4]:
attention_docs = loader_attention.load()
rag_docs = loader_rag.load()

In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [6]:
attention_chunks = text_splitter.split_documents(attention_docs)
rag_chunks = text_splitter.split_documents(rag_docs)

In [7]:
# HNSW (Hierarchical Navigable Small World) is an approximate nearest-neighbor indexing algorithm used by vector databases to perform fast similarity search.
# "cosine" -> cosine similarity
# "l2" -> euclidean distance
# "ip" -> inner product
attention_vector_store = Chroma(
    embedding_function=HuggingFaceEmbeddings(model='thenlper/gte-small', model_kwargs={"device": "cuda"}),
    persist_directory='/content/attention_db',
    collection_name='attention_collection',
    collection_metadata={"hnsw:space": "cosine"}
)

attention_vector_store.add_documents(attention_chunks)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/68.1k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_1800/529438417.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  attention_vector_store = Chroma(


['f07c2dce-cd59-4bc6-89de-e3dc612776b7',
 '6304fd7d-83ff-409f-8dd5-e618ffa61b9a',
 '8e281d49-8d4a-4041-aa01-cad0b4e206f2',
 '24a354c1-6d26-437d-afd5-4e95de7ced07',
 '03a9ed89-7de9-4bab-b013-5e8782258a53',
 '967c23ad-75bb-466a-b170-ec653f0e937a',
 '67ba6cc4-7193-45c1-ac1c-aea986bcb5b1',
 '30357a59-1f2b-4c86-8841-d437b0c05f70',
 '8df23ad7-6e09-4834-b30f-f0000fd7d524',
 '0f599016-c099-4012-a818-aa9ff55e9ec1',
 'a85e189b-a124-496b-b3a1-9e29b94d7d7d',
 '83f3b3cd-940c-4be9-be18-342bb35817e3',
 '9e1cc097-71d5-47bd-9bd1-b3eb0c26202e',
 'b07fecd6-ee93-41c5-87b0-7563f0fefd93',
 'e014f701-e713-424f-929d-630e7114e149',
 '44a11e65-1521-47d5-82db-9c17bb181510',
 '606585c7-c4a7-40fa-8a3f-5b679cb6131e',
 '9fd9cefd-58c5-4e21-bc42-69b6b01464b4',
 '444e36ad-52a7-484e-bd24-1e36e48ac997',
 '76e2c8df-4559-4bb7-8afb-b032430ab388',
 'cfd42eb4-d2e1-4a10-87d7-ecc31e355244',
 'e65c0135-4694-4813-9eb7-a5895d9c07a4',
 '5073c5a2-f582-42e0-8801-4e0a160ce396',
 '43740c2f-78e4-44df-b633-b49241ae7dfb',
 '007dc2ca-b38a-

In [8]:
rag_vector_store = Chroma(
    embedding_function=HuggingFaceEmbeddings(model='thenlper/gte-small', model_kwargs={"device": "cuda"}),
    persist_directory='/content/rag_db',
    collection_name='rag_collection',
    collection_metadata={"hnsw:space": "cosine"}
)

rag_vector_store.add_documents(rag_chunks)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

['a459dcb5-9e26-47cd-a8d6-bad6318a082a',
 'c1832711-2bc7-4bc6-a303-98161e9a06bf',
 '28461883-0927-40b7-bc51-76b5beb82e59',
 '152756fd-683d-43c7-98ac-c6a957054a0f',
 'cda9b66d-55d5-4efa-b649-8f9270270ac5',
 'b7e0c34d-c865-4b33-a5cc-941c8a955995',
 '5edc18fd-5b2c-459d-8e73-1c3f4a043c34',
 'c92f6133-05aa-4522-a380-c921ce169f59',
 'fae9a26d-c9e0-43d6-bb5e-166dc2f57600',
 'af7f0313-39b2-4a87-ba3c-cdacb1565b58',
 'b4383b87-ecf6-492d-8bcd-a8e33306c25e',
 '06583abd-104c-40bb-ae74-e547ec35c004',
 '52e2cb36-7711-42a7-8049-d6f20f6f7b0b',
 '9039f7cb-a9e7-44d7-8e2a-10e3797b964c',
 '1c48f266-ef64-4a70-8ad8-db51cd2813ee',
 'b0c7337d-3ad4-4b2a-b946-391b8a529d00',
 '4c4fc675-4272-4997-b5a7-9a1b7a71b812',
 'f5935445-4538-490f-b3c2-e7d4b7bc47f7',
 '14d952bb-c9fc-4a57-8f7c-846dc5e51ad0',
 '5b6c1615-4535-4aa8-857a-0b2f93cbb617',
 '521b07e5-2b2e-4e91-8b27-99a2f7cec99a',
 '5d72f322-67e8-48a4-a418-0bd5f6807991',
 '1bef2d73-fd56-483d-980c-3bf07bf1524b',
 '52ef65b6-1eff-44d0-8c4f-bf363ac7fdd4',
 'ead4deb2-3a73-

In [9]:
attention_retriever = attention_vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 3, "lambda_mult": 0.5})

rag_retriever = rag_vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 3, "lambda_mult": 0.5})

MergerRetriever aka Lord of the Retriever(LOTR) merges the outputs of multiple retrievers, creating a unified retrieval strategy.

Bow down to the lord!

In [10]:
lotr = MergerRetriever(retrievers=[attention_retriever, rag_retriever])

In [11]:
for chunks in lotr.invoke("What is Transformer achitecture and how does it helps in Retrieval Augmented Generation?"):
    print(chunks.page_content)

To the best of our knowledge, however, the Transformer is the ﬁrst transduction model relying
entirely on self-attention to compute representations of its input and output without using sequence-
aligned RNNs or convolution. In the following sections, we will describe the Transformer, motivate
self-attention and discuss its advantages over models such as [14, 15] and [8].
3 Model Architecture
Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 29].
retriever was trained to retrieve documents which contain answers to TriviaQA [24] questions and
Natural Questions [29]. We refer to the document index as the non-parametric memory.
2.3 Generator: BART
The generator componentpθ(yi|x,z,y 1:i−1) could be modelled using any encoder-decoder. We use
BART-large [32], a pre-trained seq2seq transformer [58] with 400M parameters. To combine the input
x with the retrieved contentz when generating from BART, we simply concatenate them. BART was
model by multipl

Sending this much trash ass messy text can make any LLM face the "Lost in Middle" situation, so we gotta clean it

In [ ]:
filter = EmbeddingsRedundantFilter(embeddings=HuggingFaceEmbeddings(model='thenlper/gte-small', model_kwargs={"device": "cuda"}))
reorder = LongContextReorder()
reranker = CohereRerank(model="rerank-v4.0-pro", top_n=2)
pipeline = DocumentCompressorPipeline(transformers=[filter, reorder, reranker])
compression_retriever = ContextualCompressionRetriever(base_compressor=pipeline, base_retriever=lotr)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [26]:
for chunks in compression_retriever.invoke("What is Transformer achitecture and how does it helps in Retrieval Augmented Generation?"):
    print(chunks.page_content)

retriever was trained to retrieve documents which contain answers to TriviaQA [24] questions and
Natural Questions [29]. We refer to the document index as the non-parametric memory.
2.3 Generator: BART
The generator componentpθ(yi|x,z,y 1:i−1) could be modelled using any encoder-decoder. We use
BART-large [32], a pre-trained seq2seq transformer [58] with 400M parameters. To combine the input
x with the retrieved contentz when generating from BART, we simply concatenate them. BART was
To the best of our knowledge, however, the Transformer is the ﬁrst transduction model relying
entirely on self-attention to compute representations of its input and output without using sequence-
aligned RNNs or convolution. In the following sections, we will describe the Transformer, motivate
self-attention and discuss its advantages over models such as [14, 15] and [8].
3 Model Architecture
Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 29].


In [46]:
# from llama_cpp import Llama
from langchain_community.llms import LlamaCpp
# from langchain_core.callbacks import StreamingStdOutCallbackHandler
from huggingface_hub import hf_hub_download

# Downloads and caches automatically
model_path = hf_hub_download(
    repo_id="TheBloke/zephyr-7B-beta-GGUF",
    filename="zephyr-7b-beta.Q4_K_M.gguf"
)

llm = LlamaCpp(
    model_path=model_path,
    n_ctx=4096,
    max_tokens=512,
    temperature=0.75,
    n_gpu_layers=35,
    n_threads=2,
    # streaming=True,
    # callbacks=[StreamingStdOutCallbackHandler()],
    streaming=False,
    verbose=False
)

llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


In [47]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [48]:
prompt = PromptTemplate.from_template("""<|system|>
You are a strict context-only assistant. You must follow these rules absolutely:
1. ONLY answer using information explicitly present in the context below.
2. If the question cannot be answered from the context, respond with ONLY: "I don't know."
3. Do NOT use any external knowledge, training data, or common sense answers.
4. Do NOT make up or infer any information not directly stated in the context.
</s>
<|user|>
Context:
{context}

Question: {question}
</s>
<|assistant|>
""")

In [49]:
parallel_chain = RunnableParallel({
    'context': RunnableSequence(compression_retriever, RunnableLambda(format_docs)),
    'question': RunnablePassthrough()
})

# output streams directly, no duplicate
main_chain = parallel_chain | prompt | llm | StrOutputParser()

In [51]:
print(main_chain.invoke("How does the Transformer architecture, specifically the attention mechanism, enable the retrieval and generation components of Retrieval Augmented Generation?"))

The Transformer architecture, which consists of stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, enables the retrieval and generation components of Retrieval Augmented Generation (RAG) through its attention mechanism.

In RAG, a query is first sent through an encoder composed of N = 6 identical layers, each with two sublayers: multi-head self-attention and point-wise, fully connected layers. The resulting context vector is then passed to the decoder, which generates a response based on the input context.

During generation, the Transformer's attention mechanism allows the model to selectively focus on specific parts of the input context as it generates each token. This mechanism enables the model to retrieve relevant information from multiple sources simultaneously, combining content from several documents to generate more accurate and diverse responses. In fact, as shown in Figure 2, the model can generate titles without depending on spec

In [50]:
print(main_chain.invoke("What is the recipe for Biryani?"))

I don't know. The provided context does not include a recipe for Biryani.
